In [1]:
import polars as pl

# Create DataFrame 
data = [{"fruit": "apple", "count": 10, "price": 0.50}, 
        {"fruit": "banana", "count": 20, "price": 0.25}]
df = pl.from_dicts(data)

# Expressions to select, filter, aggregate
sel = df.select(["fruit", "count"]) # Select columns
filt = sel.filter(pl.col("fruit") == "apple") # Filter rows
agg = filt.group_by("fruit").agg(pl.col("count").sum()) # Aggregate

print(agg)

shape: (1, 2)
┌───────┬───────┐
│ fruit ┆ count │
│ ---   ┆ ---   │
│ str   ┆ i64   │
╞═══════╪═══════╡
│ apple ┆ 10    │
└───────┴───────┘


In [ ]:
libros = {'Libro' : ['Don Quijote de la Mancha', 'Cien años de soledad', '1984', 'El principito'], 
          'Autor' : ['Miguel de Cervantes', 'Gabriel Garcia Maquez', 'George Orwell', 'Antoine de Saint-Exupéry']}
df_Json = pl.from_dicts(libros)
path = r"C:\Users\mreye\OneDrive\Documentos\Mateo\Cursos\Data engineer\libros.json"
df_Json.write_json(path)

#Leer en datos JSON y escribir en formato Parquet
df_parquet = pl.read_json(path)
df_parquet.write_parquet

<bound method DataFrame.write_parquet of shape: (4, 2)
┌──────────────────────────┬──────────────────────────┐
│ Libro                    ┆ Autor                    │
│ ---                      ┆ ---                      │
│ str                      ┆ str                      │
╞══════════════════════════╪══════════════════════════╡
│ Don Quijote de la Mancha ┆ Miguel de Cervantes      │
│ Cien años de soledad     ┆ Gabriel Garcia Maquez    │
│ 1984                     ┆ George Orwell            │
│ El principito            ┆ Antoine de Saint-Exupéry │
└──────────────────────────┴──────────────────────────┘>

In [14]:
#Unir dos conjuntos de datos y luego realizar la agregación group_by
df2 = pl.DataFrame(
    { 
        'Libro' : ['Don Quijote de la Mancha', 'Cien años de soledad', '1984', 'El principito'],
        'Año' : ['1605', '1967', '1949', '1943'],
        'Genero' : ['Novela', 'Novela', 'Ficcion', 'Fabula']
    }
)
df = df_Json.join(df2 , on='Libro', how='left')
print(df.group_by(pl.col('Genero')).agg( pl.count('Genero').alias('Numero de libros')))


shape: (3, 2)
┌─────────┬──────────────────┐
│ Genero  ┆ Numero de libros │
│ ---     ┆ ---              │
│ str     ┆ u32              │
╞═════════╪══════════════════╡
│ Novela  ┆ 2                │
│ Ficcion ┆ 1                │
│ Fabula  ┆ 1                │
└─────────┴──────────────────┘


In [15]:
#Seleccionar subconjunto de columnas, filtrar filas y añadir nueva columna

df = df.with_columns(pl.col('Año').cast(pl.Int32))
result = df.select(
    pl.col('Año'), pl.col('Libro'), Diferencia_año = 2026 - pl.col('Año')
    ).filter(
        pl.col('Año') > 1900)
result
print(df.group_by(pl.col('Genero')).agg(  pl.col('Año').std().alias('Desviacion_año')))

shape: (3, 2)
┌─────────┬────────────────┐
│ Genero  ┆ Desviacion_año │
│ ---     ┆ ---            │
│ str     ┆ f64            │
╞═════════╪════════════════╡
│ Ficcion ┆ null           │
│ Novela  ┆ 255.972655     │
│ Fabula  ┆ null           │
└─────────┴────────────────┘


In [10]:
boole = df.with_columns().filter(
    (pl.col('Año') > 1500) & (pl.col('Genero') == 'Novela')
)
boole

Libro,Autor,Año,Genero
str,str,i32,str
"""Don Quijote de la Mancha""","""Miguel de Cervantes""",1605,"""Novela"""
"""Cien años de soledad""","""Gabriel Garcia Maquez""",1967,"""Novela"""


In [2]:
#Construir una expresión para encontrar correlaciones entre columnas

estudio = pl.DataFrame({
    "Estudiante": ["Ana", "Luis", "Carla", "Diego", "Sofia", "Marco"],
    "Horas_estudio": [2, 5, 1, 4, 6, 3],
    "Calificacion": [65, 85, 55, 78, 92, 70],
    "Horas_sueño": [8, 6, 9, 7, 5, 8]
})
correlacion = estudio.select(
    correlacion_estudio_calificacion = pl.corr('Horas_estudio', 'Calificacion')
)
correlacion

correlacion_estudio_calificacion
f64
0.997592


In [8]:
desviacion = estudio.with_columns( Desviacion =  pl.col('Calificacion').std())
desviacion

Estudiante,Horas_estudio,Calificacion,Horas_sueño,Desviacion
str,i64,i64,i64,f64
"""Ana""",2,65,8,13.556056
"""Luis""",5,85,6,13.556056
"""Carla""",1,55,9,13.556056
"""Diego""",4,78,7,13.556056
"""Sofia""",6,92,5,13.556056
"""Marco""",3,70,8,13.556056


In [3]:
estudio.plot.bar(x='Calificacion', y='Horas_estudio')

alt.Chart(...)

In [11]:
import random

df3 = pl.DataFrame(
    {'numbers' : [random.randint(-200000,200000) for i in range(50000)]}
)
df3

numbers
i64
58887
82673
75011
-72591
-57684
…
32881
-150888
184494
